# ⚡ Módulo 00 - Notebook 04: Generación de Código PySpark y SQL

## 💻 Desarrollo Acelerado de ETL con IA

**Libro:** Saliendo de lo Pandito  
**Módulo:** 00 - Guía Rápida de Genie Code  
**Duración estimada:** 40 minutos  
**Dificultad:** 🟡 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Generar** código PySpark desde cero con Genie Code  
✅ **Migrar** código Pandas a PySpark automáticamente  
✅ **Optimizar** consultas SQL complejas con IA  
✅ **Aplicar** patrones de ETL (Extract, Transform, Load)  
✅ **Validar** código generado por IA  
✅ **Acelerar** desarrollo de pipelines de datos 10x

---

## 📋 Pre-requisitos

* ✅ Haber completado Notebooks 00_01, 00_02, 00_03  
* ✅ Conocimientos básicos de Python y SQL  
* ✅ Comprender qué es PySpark (se explica en Módulo 11)  
* ⚠️ **Opcional:** Experiencia con Pandas

---

## 🎯 ¿Por Qué Este Notebook?

**El desafío de Big Data:**
- PySpark tiene sintaxis diferente a Pandas
- SQL optimizado requiere conocimiento avanzado
- ETL pipelines son complejos y propensos a errores

**Con Genie Code:**
- Genera código distribuido sin memorizar APIs
- Traduce Pandas a PySpark instantáneamente
- Optimiza SQL automáticamente
- Acelera desarrollo de pipelines 10x

---

## ⚠️ Regla de Oro

> **"Genie genera, tú validas"**
> 
> Siempre revisa el código generado:
> * ¿Hace lo que necesitas?
> * ¿Es eficiente?
> * ¿Entiendes cómo funciona?
>
> Genie es un copiloto, no un autopiloto.

---

## 📖 Contenido del Notebook

1. **PySpark desde Cero** - Crear DataFrames, transformaciones básicas  
2. **Migración Pandas → PySpark** - Traducción automática  
3. **SQL Optimizado** - Consultas complejas, joins, agregaciones  
4. **Patrones ETL** - Extract, Transform, Load comunes  
5. **Validación** - Cómo revisar código generado  
6. **Casos Prácticos** - Pipelines reales de negocio

## ⚡ Generación de Código PySpark desde Cero

### 📝 Patrón de Prompt para PySpark

```
"Genera código PySpark que:
1. [ENTRADA DE DATOS] - de dónde leer
2. [TRANSFORMACIONES] - qué operaciones aplicar
3. [AGREGACIONES] - qué cálculos hacer
4. [SALIDA] - formato y destino del resultado

Compatible con Databricks Serverless."
```

---

### 🎯 Ejemplos de Prompts Efectivos

#### Ejemplo 1: Lectura y Filtrado
```
Prompt:
"Lee un archivo CSV desde /Volumes/main/default/ventas/ventas_2024.csv
Filtra filas donde region = 'LATAM' y revenue > 1000
Muestra las primeras 10 filas"

Genie genera:
```

```python
from pyspark.sql.functions import col

df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/main/default/ventas/ventas_2024.csv")
)

df_filtrado = df.filter(
    (col("region") == "LATAM") & 
    (col("revenue") > 1000)
)

df_filtrado.show(10)
```

#### Ejemplo 2: Agregación y Window Functions
```
Prompt:
"Desde la tabla catalog.schema.transacciones:
1. Agrupa por cliente_id y fecha
2. Calcula: total_compras, ticket_promedio, num_transacciones
3. Añade ranking por total_compras (descendente) dentro de cada fecha
4. Filtra solo top 10 clientes por día"

Genie genera:
```

```python
from pyspark.sql import Window
from pyspark.sql.functions import col, sum, avg, count, row_number

df = spark.table("catalog.schema.transacciones")

df_agg = df.groupBy("cliente_id", "fecha").agg(
    sum("monto").alias("total_compras"),
    avg("monto").alias("ticket_promedio"),
    count("*").alias("num_transacciones")
)

window_spec = Window.partitionBy("fecha").orderBy(col("total_compras").desc())

df_ranked = df_agg.withColumn(
    "ranking",
    row_number().over(window_spec)
)

df_top10 = df_ranked.filter(col("ranking") <= 10)

display(df_top10)
```

In [0]:
# 💻 EJERCICIO 1: Genera tu Primer Código PySpark

print("🎯 EJERCICIO: Genera código PySpark con Genie")
print("="*60)
print("\n👉 Prompt sugerido para Genie:")
print()
print('"Crea un DataFrame PySpark de ejemplo con:')
print('- 1000 filas')
print('- Columnas: user_id (entero), fecha (dates en enero 2024),')
print('  producto (A/B/C/D), cantidad (1-10), precio (10-1000)')
print('- Calcula revenue = cantidad * precio')
print('- Agrupa por producto y calcula:')
print('  * revenue total')
print('  * cantidad promedio')
print('  * número de transacciones')
print('- Ordena por revenue descendente')
print('- Muestra resultado')
print('"')
print("\n" + "="*60)
print("👇 Pega el código generado abajo y ejécutalo")

## 🔄 Migración de Pandas a PySpark

### 🎯 El Desafío

Tienes código Pandas que funciona en 100K filas, pero necesitas escalarlo a 100M filas.

**Problema:** Reescribir manualmente toma horas y es propenso a errores.

**Solución:** Genie traduce automáticamente.

---

### 📝 Patrón de Prompt de Migración

```
"Convierte este código Pandas a PySpark:

[PEGAR CÓDIGO PANDAS]

Requisitos:
- Compatible con Databricks Serverless
- Optimizado para grandes volúmenes
- Mantener la lógica de negocio exacta
- Agregar comentarios explicativos"
```

---

### 📊 Ejemplo de Migración

#### 🐼 Código Pandas Original
```python
import pandas as pd

df = pd.read_csv('ventas.csv')

# Filtrar
df_filtrado = df[df['region'] == 'LATAM']

# Crear columna calculada
df_filtrado['revenue'] = df_filtrado['cantidad'] * df_filtrado['precio']

# Agrupar
resumen = df_filtrado.groupby('producto').agg({
    'revenue': 'sum',
    'cantidad': 'mean',
    'user_id': 'count'
}).reset_index()

# Renombrar columnas
resumen.columns = ['producto', 'revenue_total', 'cantidad_promedio', 'num_ventas']

# Ordenar
resumen = resumen.sort_values('revenue_total', ascending=False)
```

#### ⚡ Código PySpark Generado por Genie
```python
from pyspark.sql.functions import col, sum, avg, count

# Leer datos
df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/main/default/ventas.csv")
)

# Filtrar
df_filtrado = df.filter(col("region") == "LATAM")

# Crear columna calculada
df_con_revenue = df_filtrado.withColumn(
    "revenue",
    col("cantidad") * col("precio")
)

# Agrupar y agregar
resumen = df_con_revenue.groupBy("producto").agg(
    sum("revenue").alias("revenue_total"),
    avg("cantidad").alias("cantidad_promedio"),
    count("user_id").alias("num_ventas")
)

# Ordenar
resumen_ordenado = resumen.orderBy(col("revenue_total").desc())

# Mostrar
display(resumen_ordenado)
```

### 🔍 Diferencias Clave Explicadas por Genie

| Pandas | PySpark | Razón |
|--------|---------|--------|
| `df[df['col'] == val]` | `df.filter(col("col") == val)` | API distribuida |
| `df['new'] = df['a'] * df['b']` | `df.withColumn("new", col("a") * col("b"))` | Inmutabilidad |
| `df.groupby().agg()` | `df.groupBy().agg()` | CamelCase en Spark |
| `df.sort_values()` | `df.orderBy()` | Nombres de métodos |
| `df.reset_index()` | No necesario | Spark no tiene índices |

In [0]:
# 💻 EJERCICIO 2: Migra Tu Código Pandas a PySpark

import pandas as pd
from datetime import datetime, timedelta
import random

print("🐼 Código Pandas Original (funciona en datos pequeños):")
print("="*60)

# Crear datos de ejemplo
fechas = [datetime(2024, 1, 1) + timedelta(days=i) for i in range(100)]
df_pandas = pd.DataFrame({
    'fecha': random.choices(fechas, k=500),
    'cliente': [f'C{i:03d}' for i in random.choices(range(50), k=500)],
    'producto': random.choices(['Laptop', 'Mouse', 'Teclado', 'Monitor'], k=500),
    'cantidad': random.choices(range(1, 10), k=500),
    'precio': random.choices([1200, 25, 80, 350], k=500)
})

# Análisis Pandas
df_pandas['revenue'] = df_pandas['cantidad'] * df_pandas['precio']
df_pandas['mes'] = pd.to_datetime(df_pandas['fecha']).dt.month

resumen_pandas = df_pandas.groupby(['mes', 'producto']).agg({
    'revenue': ['sum', 'mean'],
    'cantidad': 'sum',
    'cliente': 'nunique'
}).reset_index()

resumen_pandas.columns = ['mes', 'producto', 'revenue_total', 'revenue_promedio', 
                          'cantidad_total', 'clientes_unicos']

print("\n✅ Resultado Pandas:")
print(resumen_pandas.head(10))
print(f"\nShape: {resumen_pandas.shape}")

print("\n" + "="*60)
print("👉 TAREA: Usa Genie para convertir esto a PySpark")
print("\nPrompt sugerido:")
print('"Convierte el código Pandas de arriba a PySpark.')
print('Crea primero un DataFrame Spark desde df_pandas usando')
print('spark.createDataFrame(df_pandas), luego aplica las mismas')
print('transformaciones usando PySpark API."')

## 🚀 Optimización de Consultas SQL

### 🐢 SQL Lento vs SQL Rápido

#### ❌ Consulta Ineficiente
```sql
SELECT 
    *,
    (SELECT COUNT(*) FROM ordenes WHERE cliente_id = c.id) as num_ordenes
FROM clientes c
WHERE pais = 'Colombia'
```

**Problemas:**
- `SELECT *` trae columnas innecesarias
- Subconsulta correlacionada (ejecuta por cada fila)
- Sin índices/particiones

---

#### ✅ Consulta Optimizada (Generada por Genie)
```sql
WITH ordenes_por_cliente AS (
    SELECT 
        cliente_id,
        COUNT(*) as num_ordenes
    FROM ordenes
    GROUP BY cliente_id
)
SELECT 
    c.id,
    c.nombre,
    c.email,
    COALESCE(o.num_ordenes, 0) as num_ordenes
FROM clientes c
LEFT JOIN ordenes_por_cliente o ON c.id = o.cliente_id
WHERE c.pais = 'Colombia'
```

**Mejoras:**
- Solo columnas necesarias
- CTE pre-agrega datos
- JOIN en vez de subconsulta
- 10-100x más rápido

---

### 📝 Prompt Pattern para Optimización SQL

```
"Optimiza esta consulta SQL:

[PEGAR SQL]

Contexto:
- Tabla X tiene N millones de filas
- Columna Y está indexada
- Necesito resultado en < 10 segundos

Sugerencias de optimización:"
```

### ⚡ Optimizaciones Comunes que Genie Aplica

1. **Usar CTEs** en vez de subconsultas repetidas
2. **Filtrar early** con WHERE antes de JOIN
3. **Evitar SELECT *** y especificar columnas
4. **Usar UNION ALL** en vez de UNION (si no hay duplicados)
5. **Particionar por fecha** cuando sea aplicable
6. **Usar COALESCE** para manejar NULLs eficientemente
7. **Evitar funciones en WHERE** (no usar `WHERE YEAR(fecha) = 2024`)

## 🛠️ Patrones de ETL con Genie Code

### 📋 Pipeline ETL Típico

```
EXTRACT → TRANSFORM → LOAD
  ↓          ↓           ↓
Leer      Limpiar      Guardar
Datos     Transformar  Resultado
```

---

### 🎯 Prompt Pattern para ETL

```
"Crea un pipeline ETL que:

1. EXTRACT:
   - Lee desde [SOURCE]
   - Formato: [CSV/Parquet/Delta/JSON]

2. TRANSFORM:
   - Limpia: [nulls, duplicados, tipos]
   - Calcula: [nuevas columnas]
   - Filtra: [condiciones]
   - Agrega: [agregaciones]

3. LOAD:
   - Guarda en [DESTINATION]
   - Formato: [Parquet/Delta]
   - Modo: [overwrite/append]
   - Particiona por: [columna]

Compatible con Databricks Serverless."
```

---

### 📊 Ejemplo: Pipeline de Ventas Diarias

**Prompt a Genie:**
```
"Crea un pipeline ETL que:

1. Lee logs de ventas desde /Volumes/main/raw/ventas_logs.json
2. Limpia:
   - Elimina filas con revenue null
   - Elimina duplicados por transaction_id
   - Convierte fecha a date type
3. Transforma:
   - Calcula revenue = cantidad * precio_unitario
   - Agrega por día y producto
4. Guarda en tabla Delta: catalog.gold.ventas_diarias
   - Modo: append
   - Particiona por fecha"
```

**Código Generado:**
```python
from pyspark.sql.functions import col, to_date, sum as _sum, count

# EXTRACT
df_raw = (spark.read
    .option("multiLine", "true")
    .json("/Volumes/main/raw/ventas_logs.json")
)

# TRANSFORM
df_clean = (
    df_raw
    .filter(col("revenue").isNotNull())  # Eliminar nulls
    .dropDuplicates(["transaction_id"])  # Sin duplicados
    .withColumn("fecha", to_date(col("fecha")))  # Convertir tipo
)

df_enriched = df_clean.withColumn(
    "revenue",
    col("cantidad") * col("precio_unitario")
)

df_aggregated = (
    df_enriched
    .groupBy("fecha", "producto")
    .agg(
        _sum("revenue").alias("revenue_total"),
        _sum("cantidad").alias("cantidad_total"),
        count("*").alias("num_transacciones")
    )
)

# LOAD
df_aggregated.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("fecha") \
    .saveAsTable("catalog.gold.ventas_diarias")

print("✅ Pipeline ejecutado exitosamente")
```

## ✅ Buenas Prácticas: Revisar Código Generado

### ⚠️ Genie es Poderoso, Pero No Infalible

**Siempre revisa:**

#### 1️⃣ Lógica de Negocio
```python
# ¿Esta fórmula es correcta para tu caso?
revenue = cantidad * precio_unitario * (1 - descuento/100)

# Verifica:
# - ¿Descuento es porcentaje o decimal?
# - ¿Hay impuestos?
# - ¿Hay costos adicionales?
```

#### 2️⃣ Eficiencia
```python
# 🚨 Potencialmente lento
for producto in productos:
    df.filter(col("producto") == producto).count()

# ✅ Más eficiente
df.groupBy("producto").count().collect()
```

#### 3️⃣ Manejo de Nulls
```python
# ¿Qué pasa si hay nulls?
df.withColumn("revenue", col("cantidad") * col("precio"))

# Mejor con validación:
df.withColumn(
    "revenue",
    when(col("cantidad").isNotNull() & col("precio").isNotNull(),
         col("cantidad") * col("precio"))
    .otherwise(0)
)
```

#### 4️⃣ Compatibilidad Serverless
```python
# ❌ NO compatible con Serverless
df.rdd.map(lambda x: x[0])  # RDD no soportado

# ✅ Compatible
df.select(col("columna")).collect()  # DataFrame API
```

---

### 📝 Checklist de Revisión

Antes de ejecutar código generado:

* ☑️ ¿La lógica de negocio es correcta?
* ☑️ ¿Maneja casos edge (nulls, duplicados, valores extremos)?
* ☑️ ¿Es eficiente para el volumen de datos esperado?
* ☑️ ¿Es compatible con Databricks Serverless?
* ☑️ ¿Entiendo cómo funciona cada línea?
* ☑️ ¿Hay comentarios que expliquen partes complejas?

---

### 💬 Cómo Pedir a Genie que Explique

```
Prompt:
"Explica este código línea por línea:

[PEGAR CÓDIGO]

En particular, explica:
- ¿Qué hace la línea X?
- ¿Por qué se usa Y en lugar de Z?
- ¿Qué pasa si [edge case]?"
```

In [0]:
# 💻 EJERCICIO 3: Revisa y Mejora Código Generado

print("🔍 EJERCICIO: Revisa este código generado por IA")
print("="*60)
print("\nCódigo generado (tiene 2 problemas):")
print()
print("```python")
print("from pyspark.sql.functions import col, sum")
print()
print("# Calcular revenue total por región")
print("df = spark.table('ventas')")
print()
print("# Problema 1: No filtra fechas, escanea toda la tabla")
print("df_revenue = df.groupBy('region').agg(")
print("    sum('cantidad' * 'precio').alias('revenue_total')  # Problema 2: sintaxis incorrecta")
print(")")
print("")
print("df_revenue.show()")
print("```")

print("\n" + "="*60)
print("👉 TAREAS:")
print("\n1. Identifica los 2 problemas en el código")
print("\n2. Usa Genie con este prompt:")
print('   "Revisa este código PySpark. Tiene 2 problemas:')
print('   - No filtra por fecha reciente (solo últimos 30 días)')
print('   - Sintaxis incorrecta en la multiplicación de columnas')
print('   Corrígelo y explica los cambios."')
print("\n3. Compara el código original vs corregido")
print("\n4. Pregunta a Genie: '¿Qué otras optimizaciones recomiendas?'")
print("\n" + "="*60)

## 🎓 Conclusiones y Próximos Pasos

### ✅ Lo Que Aprendiste en Este Notebook

1. **Generar código PySpark** desde cero con prompts estructurados
2. **Migrar de Pandas a PySpark** automáticamente
3. **Optimizar consultas SQL** con asistencia de IA
4. **Implementar pipelines ETL** completos con Genie
5. **Revisar y validar** código generado por IA

---

### 💪 Checklist de Dominio

¿Puedes hacer esto con confianza?

* ☑️ Generar un pipeline PySpark completo con un prompt
* ☑️ Traducir código Pandas a PySpark en < 2 minutos
* ☑️ Identificar consultas SQL ineficientes
* ☑️ Crear ETL con extract-transform-load completo
* ☑️ Revisar código generado por IA críticamente
* ☑️ Explicar por qué una solución es mejor que otra

Si marcaste todo, 🎉 **¡Dominas generación de código Big Data con IA!**

---

### 🚀 ¿Qué Sigue?

**Has completado el Módulo 00: Guía Rápida de Genie Code**

Ahora estás listo para:

1. **Módulo 01:** [Entorno Databricks & GitHub](../01_Entorno_Databricks_Free_Edition_GitHub/)
   - Configurar tu workspace
   - Conectar con GitHub
   - Dominar notebooks y SQL Editor

2. **Módulo 02:** [Fundamentos de Pandas](../02_Fundamentos_Pandas/)
   - Aprendizaje acelerado con Genie Code
   - Ejercicios prácticos con IA como copiloto

3. **Módulo 11:** [Introducción a PySpark](../11_Introduccion_PySpark/)
   - Big Data desde día 1
   - Usando todo lo aprendido de Genie

---

### 💡 Usa Genie Durante Todo el Libro

**En cada módulo:**
* 👉 Pide a Genie que genere ejercicios adicionales
* 👉 Usa Genie para depurar tus errores
* 👉 Pide a Genie que revise tu código
* 👉 Pregunta a Genie cuando no entiendas algo

---

### 📚 Recursos Adicionales

* **Cheatsheet PySpark:** [/anexos/cheatsheets/PYSPARK_CHEATSHEET.md](#file-PYSPARK_CHEATSHEET.md)
* **Cheatsheet SQL:** [/anexos/cheatsheets/SQL_CHEATSHEET.md](#file-SQL_CHEATSHEET.md)
* **Troubleshooting:** [/anexos/troubleshooting/COMMON_ERRORS.md](#file-COMMON_ERRORS.md)
* **Documentación Genie:** [Databricks Docs](https://docs.databricks.com/en/genie/index.html)

---

<div style="background: linear-gradient(90deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🧞 Tu Copiloto de Big Data Está Listo</h3>
  <p><i>"Con Genie Code, el único límite es tu imaginación, no tu memoria de sintaxis."</i></p>
  <p style="margin-top: 15px; font-size: 1.2em;">
    <strong>➡️ <a href="../01_Entorno_Databricks_Free_Edition_GitHub/" style="color: white;">Comienza el Módulo 01</a></strong>
  </p>
</div>